# Phase 1.5 — P4, the weather-attributability ceiling (tile 32UNU)

**What fraction of the post-climatology NDVI anomaly is explainable from
meteorological forcing alone?** That number is **H1**, and it is simultaneously
**P3's denominator** — "how much was there to get". A forecast probe that
recovers 0.15 of anomaly variance means something different depending on
whether weather alone reaches 0.20 or 0.60.

**This phase reads no embeddings and loads no weights.** The inputs are the
cubes: in-cube E-OBS (8 variables on the DAILY axis), canonical NDVI, and the
manifest. `window_span_days` — half of P1's degenerate control — is recomputed
from the manifest timestamps through `encoders.pipeline.window_span_days`
rather than read from Phase 1.2's cache. CPU only, ~4 minutes.

**Two stages, and only one of them produces H1.**

| | |
|---|---|
| **Stage A** | Runs now, on the 20 single-year cubes. A leave-target-year-out climatology is NOT computable there, so it uses an explicitly named PROXY, `doy_climatology_within_fold`. Every number is labelled *within-season proxy climatology, tile-level, single year (2018), NOT the leave-year-out definition*. **It de-risks the code and establishes the controls. It is not H1.** |
| **Stage B** | Runs **iff** multi-year cubes are present. Uses `data.climatology.ndvi_climatology` (imported, never reimplemented) with `probes.cv` mode `crossed`, the only mode that agrees with a leave-year-out climatology. On this subset it prints a deferral and exits. It is **never** silently replaced by Stage A. |

**The constraint that makes Stage A honest.** The day-of-year curve is fitted on
TRAINING CUBES ONLY, inside each fold. Fitting it once on all 20 and then
cross-validating the residual leaks the test cubes into the **target
definition** — nested leakage, invisible in the output, inflating every number
*including every control*, leaving the table internally consistent and wrong.
Step 8 poisons the held-out rows and shows the fitted curve does not move.

**Why the headline is a margin and not an R².** P1 measured
`[clear_frac, window_span_days]` — two numbers, no image — decoding SEASON at
0.646–0.658 balanced accuracy, at or above every foundation model. Cloudiness
drives precipitation (a weather feature), *and* which frames survive the
clear-fraction filter, *and* which pixels the NDVI mean is taken over. So a
weather-only model can score above zero off the OBSERVATION PROCESS. The number
to read is `margin_over_control`.

**Four controls, none optional** — observation-process, day-of-year sanity,
weather+observation jointly, and a permutation null. Step 12 refuses a table
missing any of them.

**Effective n is 20 CUBES**, not 264 frames and not 4195 cells. It is on every
row of the CSV next to the R², with a fold-clustered CI.

```
My Drive/
└── NeurIPS-CCAI-2026/
    ├── data/raw/*.nc         SHARED cubes. NOT a phase.
    ├── phase1_2/ phase1_3/ phase1_4/
    └── phase1_5/             checkout
        └── phase1_5_repo.zip <- drag it here, leave it zipped
```


## Step 1: Install, then restart

CPU only. No `satlaspretrain-models`, no model weights, no embeddings. It needs
**scikit-learn, scipy and joblib**, which Colab ships.

In [1]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_5_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run the notebook against your own environment (pip install -r "
          "requirements.txt) and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy

    # torch arrives with Colab and is imported transitively by encoders/.
    if importlib.util.find_spec("torch") is None:
        !pip install torch

    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "torch, sklearn, scipy, joblib")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 11 would fail with no estimator."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

not on Colab: skipping the install and the restart.
Run the notebook against your own environment (pip install -r requirements.txt) and continue from Step 2.


## Step 2: Bootstrap

Extracts `phase1_5_repo.zip` into **its own** `phase1_5/` subfolder and resolves
`data/raw/` — the 20 cubes, shared across phases and never cleared.

The resolver between the sentinel comments is the **same block** as Phase 1.3's
and Phase 1.4's, and `tests/test_notebook_resolver.py` asserts all three are
character-identical — a second copy that is free to drift is worse than no copy.

It also resolves `EMB_IN`, the Phase 1.2 embeddings. **P4 never reads them.**
The path is printed with that said explicitly, so "this phase reads no
embeddings" is visible in the output rather than asserted in a docstring.

In [2]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "probes/cv.py",
            "probes/p1_appearance.py", "probes/p4_ceiling.py",
            "tests/test_cv_folds.py", "tests/test_p4_ceiling.py",
            "tests/conftest.py"]
ZIP_NAME = "phase1_5_repo.zip"
PHASE = "phase1_5"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p4_ceiling.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.5 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_5
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_5/ removes
        everything Phase 1.5 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz)")
print("        ^ resolved by the shared bootstrap block and DELIBERATELY UNUSED:")
print("          P4 reads no embeddings and loads no weights. window_span_days")
print("          is recomputed from the manifest, not read from this cache.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders.manifest import build_manifest
from probes import cv
from probes import p4_ceiling as p4
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"P4 at {p4.__name__}: targets {p4.TARGETS}, fold modes {p4.FOLD_MODES}, "
      f"feature sets {p4.FEATURE_SETS}, estimators {p4.ESTIMATORS}")
print(f"         model kinds {p4.MODEL_KINDS}")
print(f"         Stage B mode {p4.STAGE_B_MODE!r}, climatology harmonics "
      f"{p4.CLIMATOLOGY_HARMONICS}, doy-control harmonics "
      f"{p4.DOY_CONTROL_HARMONICS}")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

not on Colab, assuming the repo is the current directory

REPO    /Users/benji/Code/NeurIPS CCAI 2026
RAW     /Users/benji/Code/NeurIPS CCAI 2026/data/raw   (20 cubes)
EMB_IN  /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings   (100 .npz)
        ^ resolved by the shared bootstrap block and DELIBERATELY UNUSED:
          P4 reads no embeddings and loads no weights. window_span_days
          is recomputed from the manifest, not read from this cache.
RESULTS data/phase1_5/results   (this phase writes here only)
[paths] data/phase1_5: 1 file(s), 0.68 MB
[paths]   results/: 1 file(s), 0.68 MB



imports OK. canonical NDVI at data.ndvi, splits at probes.cv, modes ('cube', 'crossed', 'year', 'tile', 'spatial_block', 'temporal')
P4 at probes.p4_ceiling: targets ('cube_mean', 'cube_p90', 'cell_mean'), fold modes ('cube', 'loco', 'spatial_block'), feature sets ('weather_full8', 'weather_eowm5'), estimators ('linear', 'hgb', 'mlp')
         model kinds ('weather', 'observation', 'doy', 'weather_plus_observation', 'permutation')
         Stage B mode 'crossed', climatology harmonics 4, doy-control harmonics 6
  ok  data/ndvi.py
  ok  data/loader.py
  ok  data/paths.py
  ok  data/climatology.py
  ok  encoders/manifest.py
  ok  encoders/pipeline.py
  ok  probes/cv.py
  ok  probes/p1_appearance.py
  ok  probes/p4_ceiling.py
  ok  tests/test_cv_folds.py
  ok  tests/test_p4_ceiling.py
  ok  tests/conftest.py
helper ready: sh('<shell command>')


## Step 3: Environment check

sklearn/scipy/joblib present, and `N_JOBS` set. Parallelism changes wall-clock
only: the ridge solve is exact, HGB and the MLP carry fixed seeds with no
validation split, and the folds carry no RNG — so the CSV is identical at any
`N_JOBS`.

In [3]:
import glob, os, textwrap

import numpy as np, pandas as pd, sklearn, scipy, joblib
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__} | "
      f"joblib {joblib.__version__}")

N_JOBS = max(1, (os.cpu_count() or 2) - 1)
print(f"N_JOBS = {N_JOBS} (of {os.cpu_count()} CPUs). Wall-clock only.")

n = len(glob.glob(os.path.join(RAW, "*.nc")))
if n == 0:
    print(textwrap.dedent("""
        No cubes found. Step 4 downloads them (~15 s, 67 MB).
    """).strip())
else:
    print(f"{n} cubes at {RAW}")
print("\nNOTHING in this phase opens an .npz or a model weight.")

numpy 2.0.2 | pandas 2.3.3 | sklearn 1.6.1 | scipy 1.13.1 | joblib 1.5.3
N_JOBS = 7 (of 8 CPUs). Wall-clock only.
20 cubes at /Users/benji/Code/NeurIPS CCAI 2026/data/raw

NOTHING in this phase opens an .npz or a model weight.


## Step 4: The cubes

In [4]:
if len(glob.glob(os.path.join(RAW, "*.nc"))) >= 20:
    print("cubes already present, skipping the download")
else:
    sh(f"{PY} -m data.download_greenearthnet --out {shlex.quote(RAW)} "
       f"--n 20 --tile 32UNU")
print(f"{len(glob.glob(os.path.join(RAW, '*.nc')))} cubes at {RAW}")

cubes already present, skipping the download
20 cubes at /Users/benji/Code/NeurIPS CCAI 2026/data/raw


## Step 5: Unit tests

**This is the gate.** Expect `382 passed, 5 skipped`. The P4 suite includes the
two assertions nothing downstream could catch — that the day-of-year curve is
numerically independent of the held-out rows, and that weather is constant
across the 16 cells of a frame.

In [5]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")

$ '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python' -m pytest tests


........................................................................ [ 18%]


...................ssss...........s..................................... [ 37%]


........................................................................ [ 55%]


........................................................................ [ 74%]


........................................................................ [ 93%]
...........................                                              [100%]
=============================== warnings summary ===============================
tests/test_encoders.py::test_grid_landcover_aligns_with_the_embedding_grid
  <frozen importlib._bootstrap>:228: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
382 passed, 5 skipped, 1 warning in 55.92s


[exit 0] '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python' -m pytest tests


## Step 6: The REAL manifest, and the E-OBS join VERIFIED against the cubes

There are **two time axes** in a minicube and they are not the same axis. The
file is a ~150-step DAILY grid of which about 29 steps carry an acquisition;
`load_cube` drops the empty ones, so `original_axis_index` counts ACQUISITIONS
(it is the embedding join key) while the E-OBS series live on the DAILY axis.
`daily_axis_index` is the second one, derived from the frame's timestamp.

Until 2026-08-10 the E-OBS columns were indexed with `original_axis_index` into
a daily-axis array, so **0 of 264 rows carried the weather of their own day**
(offset 4–122 days, median 53; mean-temperature MAE 6.26 K). `assert_weather_join`
goes back to the cube, looks the day up by TIMESTAMP, and compares — the only
check that can catch it, because the manifest is internally consistent either
way.

In [6]:
from data.loader import load_cube
from encoders.manifest import assert_strata_present, assert_weather_join

SAMPLES = [load_cube(p, verbose=False)
           for p in sorted(glob.glob(os.path.join(RAW, "*.nc")))]
MANIFEST = build_manifest(SAMPLES)
assert_strata_present(MANIFEST)

print()
JOIN = assert_weather_join(MANIFEST, RAW)
assert max(JOIN["max_abs_diff"].values()) == 0.0

off = (MANIFEST.daily_axis_index - MANIFEST.original_axis_index).to_numpy()
print(f"\nMANIFEST {MANIFEST.shape} | {MANIFEST.cube_id.nunique()} cubes | "
      f"tiles {sorted(MANIFEST.tile.unique())} | years {sorted(MANIFEST.year.unique())}")
print(f"the two axes differ on {int((off != 0).sum())}/{len(MANIFEST)} rows by "
      f"{off.min()}..{off.max()} steps (median {int(np.median(off))})")

[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 122/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 122/150 timesteps with no acquisition


...

[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 120/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[manifest] 264 (cube, frame) rows over 20 cubes
[manifest] columns: ['cube_id', 'tile', 'year', 'timestamp', 'original_axis_index', 'daily_axis_index', 'day_of_year', 'pixel_bbox', 'clear_frac', 'landcover_stratum', 'landcover_dominant_frac', 'grid_landcover', 'grid_landcover_purity', 'grid_elevation_m', 'eobs_tg', 'eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tn', 'eobs_tx']
[manifest] landcover strata (per cube): {'cropland': 8, 'grassland': 6, 'tree_cover': 6}
[manifest] landcover strata (per grid cell, 320 cells over 20 cubes): {'cropland': 127, 'tree_cover': 99, 'grassland': 88, 'built_up': 5, 'bare_sparse': 1}
[manifest] cubes whose cells are NOT all one class: 19/20 -- this is the within-cube stratum contrast the per-cube label was hiding
[manifest] in-cube E-OBS joined on daily_axis_index -- the day the frame was ACQUIRED (8): ['eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tg', 'eobs_tn', 'eobs_tx']
[manifest] original_axis_index (acquisition axis)

[manifest] weather join VERIFIED against the cubes: 264 rows x 8 E-OBS variables over 20 cubes, max abs difference 0 (tolerance 1e-9)

MANIFEST (264, 22) | 20 cubes | tiles ['32UNU'] | years [np.int64(2018)]
the two axes differ on 264/264 rows by 4..122 steps (median 53)


## Step 7: What the data can and cannot support — printed BEFORE anything is fitted

Two structural facts, measured rather than assumed, that decide how the table is
read:

1. **Day-of-year vs weather.** All 264 rows land on **36 distinct days of year**
   and every one satisfies `doy % 5 == 2` — the cubes share ONE Sentinel-2 orbit
   lattice, so day-of-year is close to a 36-level categorical variable. Within a
   date the across-cube spread is 9–19% of the total for temperature, pressure
   and radiation but **77% for precipitation**, which is convective and local.
   Roughly 63% of a typical windowed weather feature is recoverable from the
   date alone. **Read the LINEAR day-of-year control as the detrend sanity
   check**; a flexible one fits a per-date mean, which here is most of a weather
   model in a different basis.

2. **Severity bins**, with edges and counts, from the reference anomaly
   distribution. This is the one full-data fit in the module and it is a
   REPORTING axis only — never a target, never a feature, never in a score. The
   anomalies actually modelled are re-derived inside every fold.

In [7]:
DATA = p4.build_p4_data(MANIFEST, RAW, verbose=True)

print()
COLLIN = p4.print_doy_weather_collinearity(MANIFEST,
                                           DATA.weather[p4.FEATURE_SETS[0]])
print()
p4.describe_estimators()
print()
for t in p4.TARGETS:
    p4.print_severity_bins(DATA.reference_anomaly[t], label=t)
    print()

[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


...

[loader] dropping 121/150 timesteps with no acquisition


[p4] targets built from 20 cubes via data.ndvi.ndvi (canonical), spatially aggregated -- never pixel-wise
[p4] cell_mean: 4224 cells -> 4195 rows (29 dropped: no valid pixel in the cell, so not an observation -- nothing is filled)


[p4] weather_full8  (264, 64) = 8 E-OBS variables x 16 aggregations x 4 windows, over the DAILY axis
[p4]   window span= 7d lag= 0d: days available min   5 median   7 max   7 | TRUNCATED on  10/264 rows ( 3.8%)
[p4]   window span=14d lag= 0d: days available min   5 median  14 max  14 | TRUNCATED on  19/264 rows ( 7.2%)
[p4]   window span=30d lag= 0d: days available min   5 median  30 max  30 | TRUNCATED on  48/264 rows (18.2%)
[p4]   window span=30d lag=30d: days available min   1 median  30 max  30 | TRUNCATED on 118/264 rows (44.7%)
[p4]   truncation is clipping at the start of the cube, never filling. Aggregates are per-day RATES so it costs precision and not scale, and days-available is deliberately NOT a feature: it correlates with position-in-cube, i.e. with time of year.


[p4] weather_eowm5  (264, 44) = 5 E-OBS variables x 11 aggregations x 4 windows, over the DAILY axis
[p4]   window span= 7d lag= 0d: days available min   5 median   7 max   7 | TRUNCATED on  10/264 rows ( 3.8%)
[p4]   window span=14d lag= 0d: days available min   5 median  14 max  14 | TRUNCATED on  19/264 rows ( 7.2%)
[p4]   window span=30d lag= 0d: days available min   5 median  30 max  30 | TRUNCATED on  48/264 rows (18.2%)
[p4]   window span=30d lag=30d: days available min   1 median  30 max  30 | TRUNCATED on 118/264 rows (44.7%)
[p4]   truncation is clipping at the start of the cube, never filling. Aggregates are per-day RATES so it costs precision and not scale, and days-available is deliberately NOT a feature: it correlates with position-in-cube, i.e. with time of year.
[p4] observation control (frame): (264, 2) ['clear_frac', 'window_span_days'] -- NO weather. window_span_days recomputed from the manifest (min 0 median 55 max 105 d), not read from the Phase 1.2 cache
[p4] obse

[p4]   hgb      HistGradientBoostingRegressor(early_stopping=False, l2_regularization=1.0,
                              max_iter=200, random_state=0)
[p4]   mlp      MLPRegressor(alpha=0.001, hidden_layer_sizes=(64, 32), max_iter=400,
             random_state=0)
[p4]   linear's alpha is D, the design width, fixed a priori and shown here at D=64; the observation control is D=2 and gets alpha=2, so margin_over_control compares two models penalised on the same scale rather than two arbitrary ones

[p4] severity bins for cube_mean -- from the REFERENCE anomaly distribution (264 rows, sd 0.0832), a reporting axis only:
[p4]   edges at quantiles [0.1, 0.3, 0.7, 0.9] = -0.1006, -0.0329, +0.0398, +0.1013
[p4]   extreme_low       27 rows (10.2%)  [-0.2664, -0.1011]
[p4]   low               52 rows (19.7%)  [-0.0994, -0.0349]
[p4]   near_normal      106 rows (40.2%)  [-0.0326, +0.0397]
[p4]   high              52 rows (19.7%)  [+0.0401, +0.1000]
[p4]   extreme_high      27 rows (10.2%)  [+0.10

## Step 8: The climatology never sees the held-out rows — EXHIBIT, not gate

The proxy climatology defines the **target**. A curve fitted outside the fold
leaks the test cubes into the target definition, which inflates every number
*including every control* — so the margins do not reveal it either, and the
table stays internally consistent while being wrong. It is the one error in this
probe that nothing downstream can catch.

Fit on the training cubes, then **add 10.0 to the held-out NDVI only** and refit.
The coefficients must be bit-identical. The gate is Step 5
(`test_the_curve_is_numerically_independent_of_the_held_out_rows`, plus its
companion proving the curve *does* move when a TRAINING row changes — a test
that can only pass would prove nothing). This cell is the visible version.

In [8]:
import inspect

sig = inspect.signature(p4.doy_climatology_within_fold)
print(f"signature: doy_climatology_within_fold{sig}")
assert list(sig.parameters)[:3] == ["day_of_year", "values", "train_idx"]
assert sig.parameters["train_idx"].default is inspect.Parameter.empty
print("  -> train_idx is REQUIRED and positional: the curve cannot be fitted on "
      "everything by omitting an argument\n")

TR = DATA.targets["cube_mean"]
doy = DATA.day_of_year[TR.row_idx]
for i, (train, test) in enumerate(p4.outer_folds(MANIFEST, "cube", k=5)):
    tp = np.flatnonzero(np.isin(TR.row_idx, train))
    ep = np.flatnonzero(np.isin(TR.row_idx, test))
    before = p4.doy_climatology_within_fold(doy, TR.y, tp)
    poisoned = TR.y.copy()
    poisoned[ep] += 10.0
    after = p4.doy_climatology_within_fold(doy, poisoned, tp)
    same = np.array_equal(before.coef, after.coef)
    # and it MUST move when a training row changes
    p2 = TR.y.copy(); p2[tp[0]] += 10.0
    moved = not np.allclose(
        before.coef, p4.doy_climatology_within_fold(doy, p2, tp).coef)
    print(f"fold {i+1}: {len(tp)} train / {len(ep)} test rows | "
          f"poison TEST -> curve identical: {same} | "
          f"poison TRAIN -> curve moves: {moved} | "
          f"absorbs {before.train_r2:.3f} of train variance")
    assert same, "THE CURVE SAW THE TEST FOLD. Nothing below this is reportable."
    assert moved
print("\nThe climatology is a function of the TRAINING index set alone.")

signature: doy_climatology_within_fold(day_of_year, values, train_idx, *, n_harmonics: 'int' = 4, period_days: 'float' = 365.25, label: 'str' = '', verbose: 'bool' = False) -> 'DoyCurve'
  -> train_idx is REQUIRED and positional: the curve cannot be fitted on everything by omitting an argument

fold 1: 211 train / 53 test rows | poison TEST -> curve identical: True | poison TRAIN -> curve moves: True | absorbs 0.303 of train variance
fold 2: 212 train / 52 test rows | poison TEST -> curve identical: True | poison TRAIN -> curve moves: True | absorbs 0.291 of train variance
fold 3: 211 train / 53 test rows | poison TEST -> curve identical: True | poison TRAIN -> curve moves: True | absorbs 0.286 of train variance
fold 4: 211 train / 53 test rows | poison TEST -> curve identical: True | poison TRAIN -> curve moves: True | absorbs 0.250 of train variance
fold 5: 211 train / 53 test rows | poison TEST -> curve identical: True | poison TRAIN -> curve moves: True | absorbs 0.244 of train var

## Step 9: Fold disjointness, and the pseudo-replicate structure

Re-derived from the manifest rather than trusted from `cv`. Then the assertion
the spec names: **weather is constant across the 16 cells of a frame**. It is a
property of the data — one cube, one day, one E-OBS reading — and if it fails
the join is wrong. A mis-indexed cell expansion changes no shape and no dtype,
which is why it is an assertion and not a comment.

That property is also *why* uncertainty is clustered at the cube level: the 16
cells of a frame are pseudo-replicates on the feature side. They add target
variance and no feature variance at all.

In [9]:
cubes = MANIFEST.cube_id.to_numpy()
for mode in p4.FOLD_MODES:
    folds = p4.outer_folds(MANIFEST, mode, k=5)
    seen = np.zeros(len(MANIFEST), dtype=int)
    for tr, te in folds:
        assert not np.intersect1d(tr, te).size
        assert not set(cubes[tr]) & set(cubes[te]), f"{mode}: a cube on both sides"
        seen[te] += 1
    print(f"{mode:<14} every manifest row tested exactly once: "
          f"{bool((seen == 1).all())}")

print()
for t in p4.TARGETS:
    p4.assert_weather_constant_across_cells(DATA, t, verbose=True)

print()
print(f"cell_mean rows: {DATA.targets['cell_mean'].n_rows} over "
      f"{len(MANIFEST)} frames over {MANIFEST.cube_id.nunique()} cubes")
print("EFFECTIVE n is 20 CUBES. Not 264 frames. Not 4195 cells.")

cube           every manifest row tested exactly once: True
loco           every manifest row tested exactly once: True
spatial_block  every manifest row tested exactly once: True

[p4] weather is CONSTANT across the cells of every frame (4195 cell_mean rows over 264 frames, 16 cells max): cell rows are PSEUDO-REPLICATES on the feature side -- they add target variance and no feature variance, so uncertainty is clustered at the cube level whatever the row count

cell_mean rows: 4195 over 264 frames over 20 cubes
EFFECTIVE n is 20 CUBES. Not 264 frames. Not 4195 cells.


## Step 10: The run

3 targets × 3 fold modes × 3 estimators × 2 feature sets × 5 model kinds =
**270 rows**, each over 5/20/5 outer folds. ~4 minutes on 7 workers.

Stage B detects the seasonal split and, on this subset, prints a deferral and
exits cleanly rather than substituting Stage A's proxy number for H1.

In [10]:
import time
t0 = time.time()
RESULTS_DF, DATA, INFO = p4.run_p4(MANIFEST, RAW, n_jobs=N_JOBS, verbose=True)
print(f"\nrun_p4: {len(RESULTS_DF)} rows in {(time.time() - t0) / 60:.1f} min")

[p4] verifying the E-OBS join against the cubes before anything is fitted -- weather is this probe's entire input


[manifest] weather join VERIFIED against the cubes: 264 rows x 8 E-OBS variables over 20 cubes, max abs difference 0 (tolerance 1e-9)
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


...

[p4]   weather_full8  permutation               D=64   R2 -5.107 [-10.866, +0.652] | vs-clim -2.412 | CRPS skill -0.870 | effective n 20 CUBES (rows 4195)


[p4]   weather_eowm5  weather                   D=44   R2 -8.250 [-18.622, +2.122] | vs-clim -3.742 | CRPS skill -1.313 | effective n 20 CUBES (rows 4195)
[p4]   weather_eowm5  observation               D=2    R2 -1.054 [-3.062, +0.955] | vs-clim -0.108 | CRPS skill -0.070 | effective n 20 CUBES (rows 4195)


[p4]   weather_eowm5  doy                       D=13   R2 -0.821 [-1.730, +0.088] | vs-clim -0.117 | CRPS skill -0.042 | effective n 20 CUBES (rows 4195)


[p4]   weather_eowm5  weather_plus_observation  D=46   R2 -3.859 [-9.852, +2.134] | vs-clim -1.494 | CRPS skill -0.622 | effective n 20 CUBES (rows 4195)


[p4]   weather_eowm5  permutation               D=44   R2 -6.495 [-14.295, +1.306] | vs-clim -3.591 | CRPS skill -1.179 | effective n 20 CUBES (rows 4195)
[p4] seasonal-split detection: years [2018] over 20 cubes; 0 cube(s) span more than one year -> multi_year=False

[p4] STAGE B DEFERRED -- pending the seasonal download.
[p4] The manifest spans [2018] over 20 cubes and 0 cube(s) span more than one year.
[p4] Stage B needs the REAL leave-target-year-out climatology
[p4]   (data.climatology.ndvi_climatology, which raises SingleYearError here BY DESIGN)
[p4] and probes.cv mode 'crossed', which holds cube AND year out jointly and
[p4] therefore agrees with that climatology's own structure. Both correctly
[p4] REFUSE a single-year manifest. Neither is weakened, and Stage A's
[p4] within-season proxy number is NOT substituted for H1: Stage A de-risks
[p4] the code and establishes the controls, and that is all it does.
[p4] H1 is produced by Stage B, on the seasonal split, and nowhere else.

## Step 11: The table's own invariants

Every one of these refuses a table rather than describing it:

- all **four controls** present for every (stage, target, fold mode, estimator,
  feature set) — without the observation control the headline margin does not
  exist, and without the permutation control there is no empirical zero;
- the two feature-set copies of a weather-free control agree digit-for-digit
  (they are the same fitted model, emitted twice so filtering the CSV to one
  feature set cannot drop a control);
- **effective n counts CUBES**, is on every row, and equals the sum of the
  per-fold cube counts;
- Stage B either ran under `crossed` with the real climatology, or was deferred
  explicitly — never silently substituted.

In [11]:
p4.assert_results_complete(RESULTS_DF)
p4.assert_stage_b_ran_or_deferred(RESULTS_DF, INFO)

print()
print("DAY-OF-YEAR SANITY CONTROL (see Step 7 for how to read it):")
d = (RESULTS_DF[RESULTS_DF.model_kind == "doy"]
     .drop_duplicates(["target", "fold_mode", "estimator"])
     .pivot_table(index=["target", "fold_mode"], columns="estimator",
                  values="r2_vs_climatology_mean"))
print(d.round(3).to_string())
lin = d["linear"].abs().max()
print(f"\nlinear day-of-year control, worst |r2| = {lin:.3f} -- the detrend "
      "removed the smooth seasonal cycle")
print("hgb/mlp are larger because a flexible learner fits a per-date mean over "
      "36 dates, which on this subset is most of a weather model in another "
      "basis. That is the collinearity from Step 7, not a failed detrend.")

print("\nPERMUTATION CONTROL -- the empirical zero of this pipeline:")
print(RESULTS_DF[RESULTS_DF.model_kind == "permutation"]
      .groupby(["target", "estimator"]).r2_vs_climatology_mean.mean()
      .round(3).to_string())
print("\nNegative, not zero: a flexible estimator on shuffled features is "
      "PENALISED rather than neutral. What matters is that it never reports "
      "positive skill, and that the real model beats it.")

[p4] results table COMPLETE: 270 rows, 1 stage(s) x 3 targets x 3 fold modes x 3 estimators x 2 feature sets x 5 model kinds
[p4] Stage B: DEFERRED, explicitly, and NOT substituted

DAY-OF-YEAR SANITY CONTROL (see Step 7 for how to read it):
estimator                  hgb  linear    mlp
target    fold_mode                          
cell_mean cube           0.069   0.001  0.019
          loco           0.043   0.002  0.030
          spatial_block  0.046   0.019 -0.117
cube_mean cube           0.168   0.007  0.111
          loco          -0.121  -0.008 -0.076
          spatial_block -0.028   0.002 -0.392
cube_p90  cube           0.421  -0.007  0.215
          loco           0.387  -0.013  0.184
          spatial_block  0.265  -0.007 -0.859

linear day-of-year control, worst |r2| = 0.019 -- the detrend removed the smooth seasonal cycle
hgb/mlp are larger because a flexible learner fits a per-date mean over 36 dates, which on this subset is most of a weather model in another basis. That is

## Step 12: The headline — the margin over the observation-process control

`margin_over_control` is `r2_vs_climatology` minus the observation control's,
for the same (stage, target, fold mode, estimator, feature set).
`r2_vs_climatology` is `1 - SSE/SSE_zero`: the climatology predicts anomaly
zero, so this is skill against the climatology itself, which is what "fraction
of post-climatology anomaly variance explained" means.

The raw R² is reported beside it. **It is not the number to quote.**

In [12]:
cols = ["model_kind", "r2_mean", "r2_ci_lo", "r2_ci_hi",
        "r2_vs_climatology_mean", "margin_over_control",
        "crps_mean", "crps_climatology_mean", "crps_skill_mean", "effective_n"]
for fs in p4.FEATURE_SETS:
    s = RESULTS_DF[(RESULTS_DF.target == "cube_mean")
                   & (RESULTS_DF.fold_mode == "cube")
                   & (RESULTS_DF.estimator == "linear")
                   & (RESULTS_DF.feature_set == fs)]
    print(f"\ncube_mean / cube / linear / {fs}")
    print(s[cols].round(4).to_string(index=False))

print("\n\nWEATHER MODEL, margin over the observation control, all cells:")
w = RESULTS_DF[RESULTS_DF.model_kind == "weather"]
print(w.pivot_table(index=["target", "fold_mode"],
                    columns=["estimator", "feature_set"],
                    values="margin_over_control").round(3).to_string())
beaten = int((w.margin_over_control <= 0).sum())
print(f"\n{beaten}/{len(w)} weather rows are AT OR BELOW the observation "
      "control: weather adds nothing there that cloud retention did not "
      "already carry.")

print("\nPER STRATUM (replication strata only; built_up has 5 cells and "
      "bare_sparse 1 over the whole subset):")
sub = RESULTS_DF[(RESULTS_DF.target == "cell_mean")
                 & (RESULTS_DF.model_kind == "weather")
                 & (RESULTS_DF.estimator == "linear")
                 & (RESULTS_DF.feature_set == "weather_full8")]
print(sub[["fold_mode"] + [f"r2_{s}" for s in p4.REPLICATION_STRATA]
          + [f"n_{s}" for s in p4.REPLICATION_STRATA]].round(3).to_string(index=False))

print("\nPER SEVERITY BIN:")
print(sub[["fold_mode"] + [f"r2_{b}" for b in p4.SEVERITY_BINS]]
      .round(3).to_string(index=False))


cube_mean / cube / linear / weather_full8
              model_kind  r2_mean  r2_ci_lo  r2_ci_hi  r2_vs_climatology_mean  margin_over_control  crps_mean  crps_climatology_mean  crps_skill_mean  effective_n
                 weather  -0.0580   -0.3796    0.2637                  0.0659               0.0383     0.0462                 0.0488           0.0492           20
             observation  -0.1051   -0.3856    0.1753                  0.0276               0.0000     0.0481                 0.0488           0.0149           20
                     doy  -0.1281   -0.3868    0.1305                  0.0073              -0.0203     0.0485                 0.0488           0.0078           20
weather_plus_observation  -0.0223   -0.3682    0.3237                  0.0947               0.0671     0.0455                 0.0488           0.0642           20
             permutation  -0.2648   -0.6031    0.0734                 -0.1197              -0.1473     0.0504                 0.0488          

## Step 13: Save, and list what this phase wrote

In [13]:
CSV = os.path.join(RESULTS, "p4_ceiling_results.csv")
RESULTS_DF.to_csv(CSV, index=False)

back = pd.read_csv(CSV)
assert back.shape == RESULTS_DF.shape, (back.shape, RESULTS_DF.shape)
p4.assert_results_complete(back)
p4.assert_stage_b_ran_or_deferred(back, INFO)
print(f"wrote {CSV}")
print(f"  {back.shape[0]} rows x {back.shape[1]} columns, "
      f"{os.path.getsize(CSV) / 1e3:.0f} kB, re-read and re-validated")
print()
describe_phase(PHASE)
print()
print("Stage A label carried on every row:")
print(" ", sorted(back.climatology_def.unique())[0])

[p4] results table COMPLETE: 270 rows, 1 stage(s) x 3 targets x 3 fold modes x 3 estimators x 2 feature sets x 5 model kinds
[p4] Stage B: DEFERRED, explicitly, and NOT substituted
wrote data/phase1_5/results/p4_ceiling_results.csv
  270 rows x 95 columns, 682 kB, re-read and re-validated

[paths] data/phase1_5: 1 file(s), 0.68 MB
[paths]   results/: 1 file(s), 0.68 MB

Stage A label carried on every row:
  within-season proxy climatology, tile-level, single year (2018), NOT the leave-year-out definition


## Phase 1.5 is done when

- [ ] Step 5: `382 passed, 5 skipped`.
- [ ] Step 6: the E-OBS join is VERIFIED against the cubes, max abs difference 0.
- [ ] Step 7: 36 distinct days of year on one orbit lattice; severity bin edges
      and counts printed before anything is fitted.
- [ ] Step 8: the day-of-year curve is bit-identical when only held-out rows are
      poisoned, and moves when a training row is.
- [ ] Step 9: weather constant across the 16 cells of every frame.
- [ ] Step 10: 270 rows.
- [ ] Step 11: all four controls present everywhere; effective n = 20 CUBES on
      every row; the linear day-of-year control near zero; the permutation
      control never positive; **Stage B DEFERRED, explicitly**.
- [ ] Step 12: the headline is `margin_over_control`, not the raw R².
- [ ] Step 13: one CSV under `data/phase1_5/results/`.

**What this phase does NOT produce.** H1. Stage A is a within-season proxy
climatology on a single year, and every row says so in its `climatology_def`
column. H1 comes from Stage B, on the seasonal split, and nowhere else.

### Re-running cleanly

```python
from data.paths import reset_phase
reset_phase("phase1_5")     # clears ONLY this phase; data/raw is untouched
```
